In [1]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
from google.colab import drive # Import drive
from io import StringIO

# --- IMPORTS FOR MODEL RE-EVALUATION ---
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    import tensorflow.keras.backend as K
    from sklearn.metrics import r2_score as sk_r2_score, mean_squared_error, mean_absolute_error
    import kagglehub

    keras.config.enable_unsafe_deserialization()

    print("TensorFlow, Scikit-learn, and KaggleHub imported successfully.")
    print("Unsafe deserialization has been ENABLED.")
except ImportError:
    print("WARNING: Key libraries not found. Stage 1 (Re-evaluation) will fail.")
    print("Please ensure tensorflow, sklearn, and kagglehub are installed.")

# --- 1. Mount Google Drive (Using Your Reference Code) ---
try:
    drive.mount('/content/drive', force_remount=True)
    print("--- Drive Mount Successful ---")
except Exception as e:
    print(f"--- ERROR Mounting Drive: {e} ---")
    print("This is often a Colab state error. Please try 'Runtime > Restart runtime...'")
    raise SystemExit("Drive mounting failed. Script cannot continue.")

# ==============================================================================
# SCRIPT STAGE 1A: DATA LOADING PIPELINES
# ==============================================================================

# --- Constants ---
FRAMES, SIZE = 28, 112
EVAL_BATCH_SIZE = 64
NUM_FRAMES = 28
IMG_HEIGHT = 112
IMG_WIDTH = 112

# --- 1) Data Pipeline for 1-Channel Models (Original) ---

def _parse_1ch_tfrecord(example):
    feature_description = {
        'clip_raw': tf.io.FixedLenFeature([], tf.string),
        'label_ef': tf.io.FixedLenFeature([], tf.float32),
    }
    example = tf.io.parse_single_example(example, feature_description)
    clip = tf.io.parse_tensor(example['clip_raw'], out_type=tf.float32)
    clip = tf.reshape(clip, [FRAMES, SIZE, SIZE, 1])
    return clip, example['label_ef']

def make_1ch_dataset(tfrecord_path, split_name, batch_size):
    tfrecord_filename = os.path.join(tfrecord_path, f"{split_name.lower()}.tfrecord")
    if not os.path.exists(tfrecord_filename):
        print(f"ERROR: 1-Ch TFRecord file not found at {tfrecord_filename}")
        return None
    dataset = tf.data.TFRecordDataset(tfrecord_filename, num_parallel_reads=tf.data.AUTOTUNE)
    dataset = dataset.map(_parse_1ch_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

def load_my_1channel_training_data():
    """Loads the 1-channel video data and labels into NumPy arrays."""
    try:
        print("Loading 1-Channel TFRecord dataset...")
        tfrecord_path = kagglehub.dataset_download('shravansaranyan/lvef-i3d-regression-tfrecord-files')
        train_ds = make_1ch_dataset(tfrecord_path, "TRAIN", EVAL_BATCH_SIZE)
        if train_ds is None:
            raise FileNotFoundError("1-Ch dataset not found.")

        print("Converting 1-Channel dataset to NumPy arrays...")
        all_features = []
        all_labels = []
        for features, labels in train_ds:
            all_features.append(features.numpy())
            all_labels.append(labels.numpy())

        features_np = np.concatenate(all_features, axis=0)
        labels_np = np.concatenate(all_labels, axis=0)

        print(f"Loaded {len(labels_np)} 1-Channel training samples.")
        return features_np, labels_np
    except Exception as e:
        print(f"FATAL ERROR loading 1-Channel data: {e}")
        return None, None

# --- 2) Data Pipeline for 2-Stream Models (Category 1 Fix) ---

def _parse_and_split_stream(proto):
    feature_description = {
        'clip_raw': tf.io.FixedLenFeature([], tf.string),
        'ef_label': tf.io.FixedLenFeature([], tf.float32),
    }
    example = tf.io.parse_single_example(proto, feature_description)
    clip_raw = example['clip_raw']
    label = example['ef_label']
    clip = tf.io.parse_tensor(clip_raw, out_type=tf.float32)
    clip = tf.ensure_shape(clip, (NUM_FRAMES, IMG_HEIGHT, IMG_WIDTH))
    clip = tf.expand_dims(clip, axis=-1)

    spatial_stream = clip[:, :, :IMG_WIDTH // 2, :]
    spatial_stream = tf.ensure_shape(spatial_stream, (28, 112, 56, 1))

    temporal_half = clip[:, :, IMG_WIDTH // 2:, :]
    temporal_stream = temporal_half[1:, :, :, :] - temporal_half[:-1, :, :, :]

    temporal_mean = tf.reduce_mean(temporal_stream, axis=[1, 2, 3], keepdims=True)
    temporal_std = tf.math.reduce_std(temporal_stream, axis=[1, 2, 3], keepdims=True)
    temporal_std = tf.maximum(temporal_std, 1e-6)
    temporal_stream = (temporal_stream - temporal_mean) / temporal_std
    temporal_stream = tf.ensure_shape(temporal_stream, (27, 112, 56, 1))

    return (spatial_stream, temporal_stream), label

def make_2stream_dataset(tfrecord_path, split_name, batch_size):
    tfrecord_filename = os.path.join(tfrecord_path, f"{split_name.lower()}.tfrecord")
    if not os.path.exists(tfrecord_filename):
        print(f"ERROR: 2-Stream TFRecord file not found at {tfrecord_filename}")
        return None
    ds = tf.data.TFRecordDataset(tfrecord_filename, num_parallel_reads=tf.data.AUTOTUNE)
    ds = ds.map(_parse_and_split_stream, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds

def load_my_2stream_training_data():
    """Loads the 2-stream video data and labels into NumPy arrays."""
    try:
        print("Downloading 2-Stream TFRecord dataset...")
        tfrecord_path = kagglehub.dataset_download('shravansaranyan/lvef-two-stream-tfrecord-files')
        train_ds = make_2stream_dataset(tfrecord_path, "train", EVAL_BATCH_SIZE)
        if train_ds is None:
            raise FileNotFoundError("2-Stream dataset not found.")

        print("Converting 2-Stream dataset to NumPy arrays...")
        all_spatial = []
        all_temporal = []
        all_labels = []
        for (spatial_batch, temporal_batch), label_batch in train_ds:
            all_spatial.append(spatial_batch.numpy())
            all_temporal.append(temporal_batch.numpy())
            all_labels.append(label_batch.numpy())

        spatial_np = np.concatenate(all_spatial, axis=0)
        temporal_np = np.concatenate(all_temporal, axis=0)
        labels_np = np.concatenate(all_labels, axis=0)

        print(f"Loaded {len(labels_np)} 2-Stream training samples.")
        return (spatial_np, temporal_np), labels_np
    except Exception as e:
        print(f"FATAL ERROR loading 2-Stream data: {e}")
        return None, None

# --- 3) Data Pipeline for 3-Channel Models (Category 3 Fix) ---

def _parse_3ch_tfrecord(example):
    feature_description = {
        'clip_raw': tf.io.FixedLenFeature([], tf.string),
        'label_ef': tf.io.FixedLenFeature([], tf.float32),
    }
    example = tf.io.parse_single_example(example, feature_description)
    clip = tf.io.parse_tensor(example['clip_raw'], out_type=tf.float32)
    clip = tf.reshape(clip, [FRAMES, SIZE, SIZE, 1])
    clip = tf.repeat(clip, 3, axis=-1)
    return clip, example['label_ef']

def make_3ch_dataset(tfrecord_path, split_name, batch_size):
    tfrecord_filename = os.path.join(tfrecord_path, f"{split_name.lower()}.tfrecord")
    if not os.path.exists(tfrecord_filename):
        print(f"ERROR: 3-Ch (I3D) TFRecord file not found at {tfrecord_filename}")
        return None
    dataset = tf.data.TFRecordDataset(tfrecord_filename, num_parallel_reads=tf.data.AUTOTUNE)
    dataset = dataset.map(_parse_3ch_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

def load_my_3channel_training_data():
    try:
        print("Loading 3-Channel TFRecord dataset...")
        tfrecord_path = kagglehub.dataset_download('shravansaranyan/lvef-i3d-regression-tfrecord-files')
        train_ds = make_3ch_dataset(tfrecord_path, "TRAIN", EVAL_BATCH_SIZE)
        if train_ds is None:
            raise FileNotFoundError("3-Ch dataset not found.")

        print("Converting 3-Channel dataset to NumPy arrays...")
        all_features = []
        all_labels = []
        for features, labels in train_ds:
            all_features.append(features.numpy())
            all_labels.append(labels.numpy())

        features_np = np.concatenate(all_features, axis=0)
        labels_np = np.concatenate(all_labels, axis=0)

        print(f"Loaded {len(labels_np)} 3-Channel training samples.")
        return features_np, labels_np
    except Exception as e:
        print(f"FATAL ERROR loading 3-Channel data: {e}")
        return None, None

# ==============================================================================
# SCRIPT STAGE 1B: CUSTOM MODEL & OBJECT HELPERS
# ==============================================================================

def get_my_custom_objects():
    @tf.keras.utils.register_keras_serializable()
    def r2_score(y_true, y_pred):
        ss_res =  K.sum(K.square(y_true - y_pred))
        ss_tot = K.sum(K.square(y_true - K.mean(y_true)))
        return 1 - ss_res/(ss_tot + K.epsilon())

    return {"r2_score": r2_score}

def build_cnn_only_model():
    frame_input_shape = (SIZE, SIZE, 3)
    base_model = keras.applications.InceptionV3(
        weights="imagenet",
        include_top=False,
        pooling="avg",
        input_shape=frame_input_shape,
    )
    fine_tune_at = 'mixed7'
    set_trainable = False
    for layer in base_model.layers:
        if layer.name == fine_tune_at:
            set_trainable = True
        if set_trainable:
            layer.trainable = True
        else:
            layer.trainable = False

    inputs = keras.Input(shape=frame_input_shape)
    preprocessed = keras.applications.inception_v3.preprocess_input(inputs)
    features = base_model(preprocessed)
    x = keras.layers.Dropout(0.5)(features)
    outputs = keras.layers.Dense(1, activation="linear", dtype='float32')(x)
    model = keras.Model(inputs, outputs, name="cnn_only_finetune")
    return model

# ==============================================================================
# SCRIPT STAGE 1C: MAIN RE-EVALUATION FUNCTION
# ==============================================================================

def reevaluate_and_update_files(base_dir, plots_dir_name, skip_list):
    """
    Walks the directory, loads models, selects the *correct* data pipeline
    for each model, re-evaluates it, and appends metrics to files.
    """

    UPDATE_HEADER = "Best Epoch (Re-evaluated) Train Set Metrics"

    # --- DEFINE MODEL CATEGORIES ---
    TWO_STREAM_MODELS = {
        'Two-Stream-L1', 'Two-Stream-B1', 'Fusion NC-B1', 'Fusion NC-B2',
        'Fusion NC-L1', 'Fusion NC-L2', 'Fusion-B1', 'Fusion-L1', 'Fusion(S)-L1'
    }
    DIFFERENTIATED_FUSION_MODELS = {
        'Fusion DI-B1', 'Fusion DI-B2', 'Fusion DI-L1', 'Fusion DI-L2',
        'Fusion DIT-1', 'Fusion DIT-2'
    }
    THREE_CHANNEL_MODELS = {
        'CNN_RNN T(O)-b(5)', 'CNN_RNN T(O)-b(7)', 'CNN_RNN-1'
    }
    PRE_COMPUTED_MODELS = {
        'CNN_RNN T(O)-a'
    }

    print("\n" + "="*60)
    print(" STAGE 1: RE-EVALUATING MODELS AND UPDATING FILES")
    print("="*60)

    # --- 1. Load All Datasets ONCE ---
    train_data_1ch, y_train_1ch = load_my_1channel_training_data()
    train_data_2stream, y_train_2stream = load_my_2stream_training_data()
    train_data_3ch, y_train_3ch = load_my_3channel_training_data()

    if train_data_1ch is None or train_data_2stream is None or train_data_3ch is None:
        print("FATAL: One or more data pipelines failed to load. Aborting Stage 1.")
        return False

    # --- 2. Get Custom Objects ---
    custom_objects = get_my_custom_objects()
    print(f"Loading models with custom objects: {list(custom_objects.keys())}")

    updated_files = 0
    skipped_already_done = 0
    failed_files = 0

    for root, dirs, files in os.walk(base_dir, topdown=True):

        if plots_dir_name in dirs:
            dirs.remove(plots_dir_name)
        dirs[:] = [d for d in dirs if d not in skip_list]

        model_name = os.path.basename(root)
        if model_name in skip_list:
            continue

        if "best_model.keras" in files and "evaluation_metrics.txt" in files:
            eval_path = os.path.join(root, "evaluation_metrics.txt")
            model_path = os.path.join(root, "best_model.keras")

            try:
                # --- A. SAFETY CHECK (Resumable) ---
                with open(eval_path, 'r') as f:
                    content = f.read()
                if UPDATE_HEADER in content:
                    skipped_already_done += 1
                    continue

                # --- B. LOAD MODEL ---
                print(f"Loading model: '{model_name}'")
                model = tf.keras.models.load_model(model_path, custom_objects=custom_objects)

                # --- C. PREDICT USING THE CORRECT DATA PIPELINE ---
                print(f"Re-evaluating '{model_name}'...")

                # Using lists for model inputs (safer than tuples)

                if model_name in TWO_STREAM_MODELS:
                    print("...using 2-Stream (split) data pipeline.")
                    # train_data_2stream is a tuple (spatial, temporal). Convert to list.
                    inputs = [train_data_2stream[0], train_data_2stream[1]]
                    y_pred_train = model.predict(inputs)
                    y_true = y_train_2stream

                elif model_name in DIFFERENTIATED_FUSION_MODELS:
                    print("...using Differentiated Fusion (1ch_video, 1ch_temporal) pipeline.")
                    # Input 1: Full 1-channel video
                    # Input 2: FULL WIDTH Temporal stream (calculated from 1ch video)

                    # Calculate temporal difference on the fly from full video
                    # Shape: (N, 28, 112, 112, 1) -> diff -> (N, 27, 112, 112, 1)
                    temporal_full = train_data_1ch[:, 1:, :, :, :] - train_data_1ch[:, :-1, :, :, :]

                    # Normalize (per video, matching typical preprocessing)
                    mean = np.mean(temporal_full, axis=(1, 2, 3), keepdims=True)
                    std = np.std(temporal_full, axis=(1, 2, 3), keepdims=True)
                    std = np.maximum(std, 1e-6)
                    temporal_full_norm = (temporal_full - mean) / std

                    inputs = [train_data_1ch, temporal_full_norm]
                    y_pred_train = model.predict(inputs)
                    y_true = y_train_1ch

                elif model_name in THREE_CHANNEL_MODELS:
                    print("...using 3-Channel data pipeline.")
                    y_pred_train = model.predict(train_data_3ch)
                    y_true = y_train_3ch

                elif model_name in PRE_COMPUTED_MODELS:
                    print("...using Category 4 (Pre-computed) pipeline.")
                    print("    Step 4a: Building feature extractor...")
                    feature_extractor_base = build_cnn_only_model().get_layer('inception_v3')
                    weights_path = os.path.join(root, "finetuned_cnn_weights.weights.h5")
                    if not os.path.exists(weights_path):
                        raise FileNotFoundError(f"Missing weights file: {weights_path}")
                    feature_extractor_base.load_weights(weights_path)

                    video_input = keras.Input(shape=(FRAMES, SIZE, SIZE, 3), name="video_input")
                    preprocessing_layer = keras.layers.Lambda(
                        keras.applications.inception_v3.preprocess_input,
                        name='inception_preprocessing'
                    )
                    preprocessed_video = keras.layers.TimeDistributed(preprocessing_layer)(video_input)
                    feature_sequences = keras.layers.TimeDistributed(feature_extractor_base)(preprocessed_video)
                    feature_extraction_model = keras.Model(video_input, feature_sequences)

                    print("    Step 4b: Extracting features from 3-Ch videos...")
                    train_features = feature_extraction_model.predict(train_data_3ch)

                    print("    Step 4c: Predicting with RNN model...")
                    y_pred_train = model.predict(train_features)
                    y_true = y_train_3ch

                else:
                    # Default: Use 1-Channel pipeline
                    print("...using 1-Channel data pipeline (default).")
                    y_pred_train = model.predict(train_data_1ch)
                    y_true = y_train_1ch

                # --- D. CALCULATE METRICS ---
                train_rmse = np.sqrt(mean_squared_error(y_true, y_pred_train))
                train_mae = mean_absolute_error(y_true, y_pred_train)
                train_r2 = sk_r2_score(y_true, y_pred_train)

                # --- E. APPEND TO FILE ---
                new_text_block = (
                    "\n\n"
                    "================================================================================\n"
                    f"{UPDATE_HEADER}\n"
                    "================================================================================\n"
                    f"- RMSE:     {train_rmse}\n"
                    f"- MAE:      {train_mae}\n"
                    f"- R2 Score: {train_r2}\n"
                    "================================================================================\n"
                )
                with open(eval_path, 'a') as f:
                    f.write(new_text_block)

                print(f"[SUCCESS] Updated '{model_name}' with re-evaluated metrics.")
                updated_files += 1

            except Exception as e:
                warnings.warn(f"Failed to process '{model_name}': {e}")
                failed_files += 1

        elif "evaluation_metrics.txt" in files and "best_model.keras" not in files:
            warnings.warn(f"'{model_name}' has metrics but no 'best_model.keras'. Skipping.")
            failed_files += 1

    print("\n--- STAGE 1 Complete ---")
    print(f"Files Newly Updated: {updated_files}")
    print(f"Files Already Up-to-Date: {skipped_already_done}")
    print(f"Files Failed: {failed_files}")
    print("="*60)
    return True # Signal success


# ==============================================================================
# SCRIPT STAGE 2: PLOTTING FUNCTIONS
# ==============================================================================

def parse_metrics_for_plotting(file_content, history_content, model_name):
    """
    Parses metrics from evaluation_metrics.txt and training_log.csv
    """
    try:
        parsed_values = {}

        # --- 1. Parse re-evaluated metrics ---
        patterns = {
            'test_rmse': re.compile(r"Test Set Metrics:.*?- RMSE:\s*([\d\.-]+)", re.DOTALL),
            'test_mae': re.compile(r"Test Set Metrics:.*?- MAE:\s*([\d\.-]+)", re.DOTALL),
            'test_r2': re.compile(r"Test Set Metrics:.*?- Sklearn R\^2:\s*([\d\.-]+)", re.DOTALL),
            'train_rmse': re.compile(r"Best Epoch \(Re-evaluated\) Train Set Metrics.*?- RMSE:\s*([\d\.-]+)", re.DOTALL),
            'train_mae': re.compile(r"Best Epoch \(Re-evaluated\) Train Set Metrics.*?- MAE:\s*([\d\.-]+)", re.DOTALL),
            'train_r2': re.compile(r"Best Epoch \(Re-evaluated\) Train Set Metrics.*?- R2 Score:\s*([\d\.-]+)", re.DOTALL),
        }

        all_found = True
        for key, pattern in patterns.items():
            match = pattern.search(file_content)
            if match:
                parsed_values[key] = float(match.group(1))
            else:
                warnings.warn(f"Parse Warning (Plot): Could not find '{key}' for model '{model_name}'.")
                all_found = False
                break

        if not all_found:
            return None

        # --- 2. Parse logged loss from training_log.csv ---
        history_df = pd.read_csv(StringIO(history_content))

        if 'val_rmse' not in history_df.columns:
            warnings.warn(f"Parse Warning (Plot): 'val_rmse' not in training_log.csv for '{model_name}'.")
            return None

        best_epoch_idx = history_df['val_rmse'].idxmin()
        best_epoch_data = history_df.loc[best_epoch_idx]

        parsed_values['train_loss'] = float(best_epoch_data['loss'])
        parsed_values['val_loss'] = float(best_epoch_data['val_loss'])

        # --- 3. Return all differences ---
        return {
            'model_name': model_name,
            'rmse_diff': parsed_values['test_rmse'] - parsed_values['train_rmse'],
            'mae_diff': parsed_values['test_mae'] - parsed_values['train_mae'],
            'r2_diff': parsed_values['test_r2'] - parsed_values['train_r2'],
            'loss_diff': parsed_values['val_loss'] - parsed_values['train_loss']
        }

    except Exception as e:
        warnings.warn(f"Critical Parsing Error (Plot) for '{model_name}': {e}. Skipping.")
        return None

def collect_all_metrics_for_plotting(base_dir, plots_dir_name, skip_list):
    """
    Walks the base directory to find and parse 'evaluation_metrics.txt'
    AND 'training_log.csv'.
    """
    metrics_data = []
    skipped_folders = []

    if skip_list is None:
        skip_list = set()

    print(f"\nWalking directory to collect data for plots: {base_dir}\n")

    for root, dirs, files in os.walk(base_dir, topdown=True):
        if plots_dir_name in dirs:
            dirs.remove(plots_dir_name)
        model_name = os.path.basename(root)
        if model_name in skip_list:
            skipped_folders.append((model_name, "Manually skipped"))
            dirs[:] = []
            continue

        # Look for training_log.csv (name is now standardized)
        eval_file_present = "evaluation_metrics.txt" in files
        history_file_present = "training_log.csv" in files

        if eval_file_present and history_file_present:
            eval_path = os.path.join(root, "evaluation_metrics.txt")
            history_path = os.path.join(root, "training_log.csv")

            try:
                with open(eval_path, 'r') as f:
                    eval_content = f.read()
                with open(history_path, 'r') as f:
                    history_content = f.read()

                metrics = parse_metrics_for_plotting(eval_content, history_content, model_name)

                if metrics:
                    metrics_data.append(metrics)
                else:
                    skipped_folders.append((model_name, "Plot parsing failed (check warnings)"))

            except Exception as e:
                warnings.warn(f"Could not read files for {model_name}: {e}")
                skipped_folders.append((model_name, f"File read error: {e}"))
            dirs[:] = []

        elif "best_model.keras" in files:
            if not eval_file_present:
                skipped_folders.append((model_name, "evaluation_metrics.txt not found"))
            if not history_file_present:
                skipped_folders.append((model_name, "training_log.csv not found"))
            dirs[:] = []

    return metrics_data, skipped_folders

def plot_metrics_diff(df, plots_dir):
    """
    Generates and saves four bar plots for the metric differences.
    """
    FIG_SIZE           = (20, 10)
    TITLE_FONT_SIZE    = 18
    LABEL_FONT_SIZE    = 14
    TICK_FONT_SIZE     = 10
    XTICK_ROTATION     = 45
    XTICK_HA           = 'right'
    GRID_ALPHA         = 0.5
    GRID_LINESTYLE     = '--'
    COLOR_OVERFIT      = 'salmon'
    COLOR_GOOD_GEN     = 'mediumseagreen'

    try:
        # --- Plot 1: RMSE Difference (Test - Train) ---
        print("Generating RMSE plot...")
        plt.figure(figsize=FIG_SIZE)
        colors = [COLOR_OVERFIT if x > 0 else COLOR_GOOD_GEN for x in df['rmse_diff']]
        plt.bar(df['model_name'], df['rmse_diff'], color=colors)
        plt.axhline(0, color='black', linestyle=GRID_LINESTYLE)
        plt.title('Generalization Gap: RMSE (Test - Train)', fontsize=TITLE_FONT_SIZE)
        plt.ylabel('RMSE Difference', fontsize=LABEL_FONT_SIZE)
        plt.xticks(rotation=XTICK_ROTATION, ha=XTICK_HA, fontsize=TICK_FONT_SIZE)
        plt.yticks(fontsize=TICK_FONT_SIZE)
        plt.grid(axis='y', linestyle=GRID_LINESTYLE, alpha=GRID_ALPHA)
        plt.tight_layout()
        plot_path = os.path.join(plots_dir, 'rmse_generalization_gap_plot.png')
        plt.savefig(plot_path)
        plt.close()

        # --- Plot 2: MAE Difference (Test - Train) ---
        print("Generating MAE plot...")
        plt.figure(figsize=FIG_SIZE)
        colors = [COLOR_OVERFIT if x > 0 else COLOR_GOOD_GEN for x in df['mae_diff']]
        plt.bar(df['model_name'], df['mae_diff'], color=colors)
        plt.axhline(0, color='black', linestyle=GRID_LINESTYLE)
        plt.title('Generalization Gap: MAE (Test - Train)', fontsize=TITLE_FONT_SIZE)
        plt.ylabel('MAE Difference', fontsize=LABEL_FONT_SIZE)
        plt.xticks(rotation=XTICK_ROTATION, ha=XTICK_HA, fontsize=TICK_FONT_SIZE)
        plt.yticks(fontsize=TICK_FONT_SIZE)
        plt.grid(axis='y', linestyle=GRID_LINESTYLE, alpha=GRID_ALPHA)
        plt.tight_layout()
        plot_path = os.path.join(plots_dir, 'mae_generalization_gap_plot.png')
        plt.savefig(plot_path)
        plt.close()

        # --- Plot 3: R² Difference (Test - Train) ---
        print("Generating R² plot...")
        plt.figure(figsize=FIG_SIZE)
        colors = [COLOR_OVERFIT if x < 0 else COLOR_GOOD_GEN for x in df['r2_diff']]
        plt.bar(df['model_name'], df['r2_diff'], color=colors)
        plt.axhline(0, color='black', linestyle=GRID_LINESTYLE)
        plt.title('Generalization Gap: R² (Test - Train)', fontsize=TITLE_FONT_SIZE)
        plt.ylabel('R² Difference', fontsize=LABEL_FONT_SIZE)
        plt.xticks(rotation=XTICK_ROTATION, ha=XTICK_HA, fontsize=TICK_FONT_SIZE)
        plt.yticks(fontsize=TICK_FONT_SIZE)
        plt.grid(axis='y', linestyle=GRID_LINESTYLE, alpha=GRID_ALPHA)
        plt.tight_layout()
        plot_path = os.path.join(plots_dir, 'r2_generalization_gap_plot.png')
        plt.savefig(plot_path)
        plt.close()

        # --- Plot 4: Loss Difference (Val - Train) ---
        print("Generating Loss plot...")
        plt.figure(figsize=FIG_SIZE)
        colors = [COLOR_OVERFIT if x > 0 else COLOR_GOOD_GEN for x in df['loss_diff']]
        plt.bar(df['model_name'], df['loss_diff'], color=colors)
        plt.axhline(0, color='black', linestyle=GRID_LINESTYLE)
        plt.title('Generalization Gap: Loss (Val - Train, from Best Epoch)', fontsize=TITLE_FONT_SIZE)
        plt.ylabel('Loss Difference (Val - Train)', fontsize=LABEL_FONT_SIZE)
        plt.xticks(rotation=XTICK_ROTATION, ha=XTICK_HA, fontsize=TICK_FONT_SIZE)
        plt.yticks(fontsize=TICK_FONT_SIZE)
        plt.grid(axis='y', linestyle=GRID_LINESTYLE, alpha=GRID_ALPHA)
        plt.tight_layout()
        plot_path = os.path.join(plots_dir, 'loss_generalization_gap_plot.png')
        plt.savefig(plot_path)
        plt.close()

        print(f"\nSuccessfully generated 4 plots in: {plots_dir}")

    except Exception as e:
        print(f"\n--- ERROR during plot generation ---")
        print(f"An error occurred: {e}")
        warnings.warn(f"Plot generation failed: {e}")

# ==============================================================================
# SCRIPT MAIN EXECUTION
# ==============================================================================

def main():
    # Define Paths
    MODELS_BASE_DIR = "/content/drive/My Drive/LVEF Prediction Models"
    PLOTS_DIR_NAME = "Plots"
    PLOTS_DIR_PATH = os.path.join(MODELS_BASE_DIR, PLOTS_DIR_NAME)

    MODELS_TO_SKIP = {
        "Fusion DI-2",
        "I3D Single-class D-1",
        "I3D Single-class D-2",
        "I3D Single-class D-5",
        "I3D Single-class M-1 (2)",
        "Appendix Models",
        "Unused Models",
        "Fusion(Z)-L1",
        "CNN_RNN Scratch(GRU)-Mixed1"
    }
    print(f"--- Will manually skip {len(MODELS_TO_SKIP)} specified folders ---")

    # --- STAGE 1: RE-EVALUATE AND UPDATE FILES ---
    stage_1_success = reevaluate_and_update_files(
        MODELS_BASE_DIR,
        PLOTS_DIR_NAME,
        MODELS_TO_SKIP
    )

    if not stage_1_success:
        print("\n--- WARNING: Stage 1 (Re-evaluation) failed. ---")
        print("The script cannot proceed to plotting.")
        print("Please review errors from Stage 1 and re-run.")
        print("--- Script Finished ---")
        return

    # --- STAGE 2: GENERATE PLOTS ---
    print("\n" + "="*60)
    print(" STAGE 2: GENERATING PLOTS FROM UPDATED FILES")
    print("="*60)

    try:
        os.makedirs(PLOTS_DIR_PATH, exist_ok=True)
        print(f"Ensured output directory exists: {PLOTS_DIR_PATH}")
    except Exception as e:
        print(f"FATAL: Could not create output directory: {e}")
        return

    # Collect and Parse Metrics
    metrics_data, skipped_folders = collect_all_metrics_for_plotting(
        MODELS_BASE_DIR,
        PLOTS_DIR_NAME,
        MODELS_TO_SKIP
    )

    if not metrics_data:
        print("\nNo valid model data was found (missing evaluation_metrics.txt or training_log.csv).")
        if skipped_folders:
            print("\n--- Folders Skipped During Plotting ---")
            for folder, reason in skipped_folders:
                print(f"- {folder}: {reason}")
        return

    # Create and Save DataFrame
    try:
        df = pd.DataFrame(metrics_data)
        df = df.sort_values(by='model_name').reset_index(drop=True)

        print("\n" + "="*50)
        print("      MODEL GENERALIZATION GAPS (Test - Train)")
        print("="*50)
        with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
            print(df)
        print("="*50)

        csv_path = os.path.join(PLOTS_DIR_PATH, 'model_generalization_gaps.csv')
        df.to_csv(csv_path, index=False)
        print(f"\nDataFrame saved to: {csv_path}")

    except Exception as e:
        print(f"\nFATAL: Could not create or save DataFrame: {e}")
        return

    # Generate Plots
    plot_metrics_diff(df, PLOTS_DIR_PATH)

    # Final Report of Skipped Folders
    if skipped_folders:
        print("\n--- Summary of Skipped Folders (Plotting Stage) ---")
        for folder, reason in skipped_folders:
            print(f"- {folder}: {reason}")

    print("\n--- Script Finished ---")

if __name__ == "__main__":
    warnings.simplefilter(action='default', category=UserWarning)
    main()

TensorFlow, Scikit-learn, and KaggleHub imported successfully.
Unsafe deserialization has been ENABLED.
Mounted at /content/drive
--- Drive Mount Successful ---
--- Will manually skip 9 specified folders ---

 STAGE 1: RE-EVALUATING MODELS AND UPDATING FILES
Loading 1-Channel TFRecord dataset...
Using Colab cache for faster access to the 'lvef-i3d-regression-tfrecord-files' dataset.
Converting 1-Channel dataset to NumPy arrays...
Loaded 7465 1-Channel training samples.
Using Colab cache for faster access to the 'lvef-two-stream-tfrecord-files' dataset.
Converting 2-Stream dataset to NumPy arrays...
Loaded 7465 2-Stream training samples.
Loading 3-Channel TFRecord dataset...
Using Colab cache for faster access to the 'lvef-i3d-regression-tfrecord-files' dataset.
Converting 3-Channel dataset to NumPy arrays...
Loaded 7465 3-Channel training samples.
Loading models with custom objects: ['r2_score']
Loading model: 'Fusion DI-B1'


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 134 variables whereas the saved optimizer has 138 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Re-evaluating 'Fusion DI-B1'...
...using Differentiated Fusion (1ch_video, 1ch_temporal) pipeline.
234/234 ━━━━━━━━━━━━━━━━━━━━ 22s 60ms/step
[SUCCESS] Updated 'Fusion DI-B1' with re-evaluated metrics.
Loading model: 'Fusion DI-B2'


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 134 variables whereas the saved optimizer has 138 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Re-evaluating 'Fusion DI-B2'...
...using Differentiated Fusion (1ch_video, 1ch_temporal) pipeline.
234/234 ━━━━━━━━━━━━━━━━━━━━ 15s 52ms/step
[SUCCESS] Updated 'Fusion DI-B2' with re-evaluated metrics.
Loading model: 'Fusion DI-L1'


...

Re-evaluating 'Fusion DI-L2'...
...using Differentiated Fusion (1ch_video, 1ch_temporal) pipeline.
234/234 ━━━━━━━━━━━━━━━━━━━━ 19s 61ms/step
[SUCCESS] Updated 'Fusion DI-L2' with re-evaluated metrics.
Loading model: 'Fusion DIT-1'


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 114 variables whereas the saved optimizer has 118 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Re-evaluating 'Fusion DIT-1'...
...using Differentiated Fusion (1ch_video, 1ch_temporal) pipeline.
234/234 ━━━━━━━━━━━━━━━━━━━━ 14s 49ms/step
[SUCCESS] Updated 'Fusion DIT-1' with re-evaluated metrics.
Loading model: 'Fusion DIT-2'


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 118 variables whereas the saved optimizer has 122 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Re-evaluating 'Fusion DIT-2'...
...using Differentiated Fusion (1ch_video, 1ch_temporal) pipeline.
234/234 ━━━━━━━━━━━━━━━━━━━━ 17s 55ms/step
[SUCCESS] Updated 'Fusion DIT-2' with re-evaluated metrics.

--- STAGE 1 Complete ---
Files Newly Updated: 6
Files Already Up-to-Date: 39
Files Failed: 0

 STAGE 2: GENERATING PLOTS FROM UPDATED FILES
Ensured output directory exists: /content/drive/My Drive/LVEF Prediction Models/Plots

Walking directory to collect data for plots: /content/drive/My Drive/LVEF Prediction Models


      MODEL GENERALIZATION GAPS (Test - Train)
                      model_name  rmse_diff  mae_diff   r2_diff  loss_diff
0        CNN_RNN Scratch(GRU)-B1   1.181414  0.755780 -0.132666  22.179634
1        CNN_RNN Scratch(GRU)-L1  -0.176738 -0.222056  1.274839  -2.895950
2       CNN_RNN Scratch(LSTM)-B1   1.525903  1.036503 -0.152782   9.412384
3       CNN_RNN Scratch(LSTM)-L1   0.147477  0.138339 -0.035430 -13.547958
4   CNN_RNN Scratch(LSTM)-Mixed1   0.644587  0.519407 